# 簇级链路（单日演示）
本 Notebook 复用现有数据与模型：先导出股票嵌入，再做 KMeans 聚类、簇画像、簇级预测。
目标是先跑通最小闭环，再扩展到多日与回测。

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from sklearn.cluster import KMeans

from models.embedding_model import load_embedding_model, extract_embeddings
from models.cluster_predictor import (
    build_cluster_samples,
    ClusterReturnPredictor,
    train_cluster_predictor,
    predict_cluster_returns,
)


def set_seed(seed_value):
    """设置随机种子。"""
    torch.manual_seed(seed_value)
    np.random.seed(seed_value)
    print("[状态] 随机种子已设置")


def get_device():
    """选择设备。"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # 运行设备
    print(f"[状态] 当前设备：{device}")
    return device

In [ ]:
def load_returns_csv(path):
    """读取收益率矩阵。"""
    if not os.path.exists(path):
        raise FileNotFoundError("收益率文件不存在，请检查 data/daily_returns.csv")
    dataframe = pd.read_csv(path)  # 原始收益率表
    date_column = "trade_date" if "trade_date" in dataframe.columns else dataframe.columns[0]  # 日期列名
    dates = pd.to_datetime(dataframe[date_column], errors="coerce")  # 日期序列
    returns = dataframe.drop(columns=[date_column], errors="ignore")  # 收益率矩阵
    returns.columns = [str(column) for column in returns.columns]
    returns = returns.apply(pd.to_numeric, errors="coerce")
    returns.index = dates
    print(f"[状态] 收益率面板加载完成，shape={returns.shape}")
    return returns


def load_industry_mapping(path, stock_codes):
    """读取行业映射并对齐。"""
    if not os.path.exists(path):
        raise FileNotFoundError("行业映射文件不存在，请检查 data/industry_mapping.csv")
    dataframe = pd.read_csv(path)  # 行业映射表
    dataframe["ts_code"] = dataframe["ts_code"].astype(str)
    dataframe = dataframe[dataframe["ts_code"].isin(stock_codes)].copy()
    print(f"[状态] 行业映射加载完成，shape={dataframe.shape}")
    return dataframe


def load_concept_mapping(data_dir, stock_codes):
    """读取概念映射并对齐。"""
    candidates = [
        os.path.join(data_dir, "stock_concept.csv"),
        os.path.join(data_dir, "concept_mapping.csv"),
        os.path.join(data_dir, "stock_concepts.csv"),
    ]
    path = next((p for p in candidates if os.path.exists(p)), None)  # 概念文件路径
    if path is None:
        print("[状态] 未找到概念映射文件，将使用空概念边")
        return pd.DataFrame(columns=["ts_code", "concept"])
    dataframe = pd.read_csv(path)  # 概念映射表
    if "ts_code" not in dataframe.columns:
        raise ValueError("概念映射缺少 ts_code 列，请检查 CSV")
    concept_column = next((c for c in dataframe.columns if c != "ts_code"), None)  # 概念列名
    if concept_column is None:
        return pd.DataFrame(columns=["ts_code", "concept"])
    dataframe = dataframe[["ts_code", concept_column]].copy()
    dataframe.columns = ["ts_code", "concept"]
    dataframe["ts_code"] = dataframe["ts_code"].astype(str)
    dataframe = dataframe[dataframe["ts_code"].isin(stock_codes)].copy()
    print(f"[状态] 概念映射加载完成，shape={dataframe.shape}")
    return dataframe

In [ ]:
def zscore_by_column(dataframe):
    """按列标准化。"""
    mean_value = dataframe.mean(axis=0)  # 列均值
    std_value = dataframe.std(axis=0).replace(0, np.nan)  # 列标准差
    return (dataframe - mean_value) / std_value


def build_features_from_window(window_dataframe):
    """用历史窗口构造特征。"""
    momentum_20 = (1 + window_dataframe.tail(20)).prod() - 1  # 20 日动量
    mean_5 = window_dataframe.tail(5).mean()  # 5 日均值
    mean_10 = window_dataframe.tail(10).mean()  # 10 日均值
    volatility_20 = window_dataframe.tail(20).std(ddof=0)  # 20 日波动
    volatility_60 = window_dataframe.tail(60).std(ddof=0)  # 60 日波动
    last_return = window_dataframe.tail(1).iloc[0]  # 最新收益
    feature_frame = pd.DataFrame({
        "mom20": momentum_20,
        "mean5": mean_5,
        "mean10": mean_10,
        "vol20": volatility_20,
        "vol60": volatility_60,
        "last_ret": last_return,
    })
    return zscore_by_column(feature_frame)


def build_stock2index(stock_codes):
    """生成股票到索引的映射。"""
    return {code: index for index, code in enumerate(stock_codes)}


def build_group_edges(mapping_df, stock2index, stock_column, group_column, max_neighbors):
    """按分组构造边。"""
    edge_pairs = set()  # 边集合
    grouped = mapping_df.groupby(group_column)[stock_column].apply(list).to_dict()  # 分组字典
    for members in grouped.values():
        members = [code for code in members if code in stock2index]
        for source in members:
            peers = [code for code in members if code != source]
            if len(peers) > max_neighbors:
                peers = peers[:max_neighbors]
            for target in peers:
                edge_pairs.add((stock2index[source], stock2index[target]))
    if not edge_pairs:
        print(f"[状态] {group_column} 分组未生成边")
        return torch.empty((2, 0), dtype=torch.long)
    print(f"[状态] {group_column} 分组边数={len(edge_pairs)}")
    return torch.tensor(sorted(edge_pairs), dtype=torch.long).t().contiguous()


def build_corr_edges(window_dataframe, top_neighbor_count):
    """构造相关性边。"""
    values = window_dataframe.to_numpy(dtype=float)  # 收益矩阵
    if values.shape[0] < 5:
        print("[状态] 相关性窗口不足，跳过相关性边")
        return torch.empty((2, 0), dtype=torch.long)
    corr = np.corrcoef(values.T)  # 相关系数矩阵
    edge_pairs = set()  # 边集合
    for i in range(corr.shape[0]):
        row = corr[i].copy()
        row[i] = -np.inf
        idx = np.argsort(np.abs(row))[-top_neighbor_count:]
        for j in idx:
            if np.isfinite(row[j]):
                edge_pairs.add((i, j))
    if not edge_pairs:
        print("[状态] 未生成相关性边")
        return torch.empty((2, 0), dtype=torch.long)
    print(f"[状态] 相关性边数={len(edge_pairs)}")
    return torch.tensor(sorted(edge_pairs), dtype=torch.long).t().contiguous()


def build_data_for_index(returns, industry_df, concept_df, stock_codes, time_index, lookback, top_neighbor_count):
    """按日期索引构建图数据。"""
    print("[状态] 开始构建当日图数据...")
    window_dataframe = returns.iloc[time_index - lookback:time_index]  # 历史窗口
    feature_frame = build_features_from_window(window_dataframe).reindex(stock_codes).fillna(0.0)  # 节点特征
    stock2index = build_stock2index(stock_codes)  # 索引映射
    industry_edge = build_group_edges(industry_df, stock2index, "ts_code", "industry", max_neighbors=20)
    concept_edge = build_group_edges(concept_df, stock2index, "ts_code", "concept", max_neighbors=20)
    corr_edge = build_corr_edges(window_dataframe, top_neighbor_count=top_neighbor_count)
    edge_index = torch.cat([industry_edge, concept_edge, corr_edge], dim=1)
    edge_type = torch.cat([
        torch.zeros(industry_edge.size(1), dtype=torch.long),
        torch.ones(concept_edge.size(1), dtype=torch.long),
        torch.full((corr_edge.size(1),), 2, dtype=torch.long),
    ])
    data = Data(x=torch.tensor(feature_frame.to_numpy(dtype=float), dtype=torch.float), edge_index=edge_index, edge_type=edge_type)
    print("[状态] 图数据构建完成")
    return data

In [ ]:
def get_cluster_codes(stock_codes, cluster_labels, cluster_id):
    """获取某个簇内的股票代码。"""
    return [code for code, label in zip(stock_codes, cluster_labels) if label == cluster_id]


def get_top_industry(industry_df, cluster_codes):
    """找出簇内占比最高的行业。"""
    if not cluster_codes:
        return "EMPTY"
    subset = industry_df[industry_df["ts_code"].isin(cluster_codes)]
    if subset.empty:
        return "UNKNOWN"
    return subset["industry"].value_counts().idxmax()


def build_cluster_profile(stock_codes, cluster_labels, industry_df, next_returns):
    """构造簇画像表。"""
    print("[状态] 开始生成簇画像...")
    profile_rows = []  # 画像行列表
    for cluster_id in sorted(set(cluster_labels)):
        cluster_codes = get_cluster_codes(stock_codes, cluster_labels, cluster_id)  # 簇内股票
        cluster_returns = [next_returns[i] for i, label in enumerate(cluster_labels) if label == cluster_id]  # 簇内收益
        mean_return = float(np.nanmean(cluster_returns)) if cluster_returns else float("nan")
        top_industry = get_top_industry(industry_df, cluster_codes)
        profile_rows.append({
            "cluster_id": int(cluster_id),
            "size": int(len(cluster_codes)),
            "mean_next_return": mean_return,
            "top_industry": top_industry,
        })
    profile = pd.DataFrame(profile_rows)
    print("[状态] 簇画像生成完成")
    return profile

In [ ]:
# 单日演示主流程
data_directory = "data"  # 数据目录
returns_path = os.path.join(data_directory, "daily_returns.csv")  # 收益率文件路径
industry_path = os.path.join(data_directory, "industry_mapping.csv")  # 行业映射路径
model_path = "best_model.pt"  # 模型路径
lookback = 60  # 回看窗口
top_neighbor_count = 20  # 相关性邻居数
cluster_count = 20  # 聚类簇数
seed_value = 42  # 随机种子
hidden_channels = 64  # 隐层维度
set_seed(seed_value)
device = get_device()

returns = load_returns_csv(returns_path)
stock_codes = list(returns.columns)  # 股票代码列表
industry_df = load_industry_mapping(industry_path, stock_codes)
concept_df = load_concept_mapping(data_directory, stock_codes)
if len(returns) <= lookback:
    raise ValueError("收益率样本不足，无法构建窗口，请先补充数据")
time_index = min(lookback, len(returns) - 1)  # 选定日期索引
print(f"[状态] 使用日期索引={time_index}，日期={returns.index[time_index].date()}")

data = build_data_for_index(returns, industry_df, concept_df, stock_codes, time_index, lookback, top_neighbor_count)
model = load_embedding_model(model_path, in_channels=6, hidden_channels=hidden_channels, num_relations=3, device=device)
embeddings = extract_embeddings(model, data, device)
print(f"[状态] 嵌入形状={embeddings.shape}")

print("[状态] 开始 KMeans 聚类...")
kmeans = KMeans(n_clusters=cluster_count, random_state=seed_value, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)
print("[状态] 聚类完成")

next_returns = returns.iloc[time_index].to_numpy(dtype=float)  # 下一期收益（单日演示用）
profile = build_cluster_profile(stock_codes, cluster_labels, industry_df, next_returns)
profile_path = "cluster_profile.csv"  # 簇画像输出路径
profile.to_csv(profile_path, index=False)
print(f"[状态] 簇画像已保存 -> {profile_path}")
display(profile.head())

cluster_features, cluster_labels_mean, cluster_id_list = build_cluster_samples(embeddings, cluster_labels, next_returns)
predictor = ClusterReturnPredictor(input_dim=cluster_features.shape[1], hidden_dim=32, dropout_rate=0.1)
predictor = train_cluster_predictor(predictor, cluster_features, cluster_labels_mean, device, epochs=30, batch_size=32, learning_rate=1e-3)
cluster_predictions = predict_cluster_returns(predictor, cluster_features, device)
prediction_table = pd.DataFrame({
    "cluster_id": cluster_id_list,
    "pred_return": cluster_predictions,
}).sort_values("pred_return", ascending=False)
print("[状态] Top-K 簇预测结果：")
display(prediction_table.head(5))